# Part 1: Getting Census Data

In this part we will:
- Connect to the U.S. Census Bureau API and download data
- Load it into a table (called a DataFrame)
- Clean up one column so it is ready for maps later

**Before you start:** Look up the variable code(s) you want at  
https://api.census.gov/data/2023/acs/acs5/profile/variables.html  
Use **Ctrl+F** to search. Example code: `DP04_0058E` (households without a vehicle).

---

## 1.1 — Import Libraries

Libraries are pre-built tools we borrow. Run this cell first every time.

In [ ]:
import requests       # for downloading data from the internet
import pandas as pd    # for working with tables of data

## 1.2 — Set Your Query Options

Change the values in this cell to match what you want to download.

> **Stick with `tract` for this workshop** — later steps are built around tracts.

Find your state FIPS code here: https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt

In [ ]:
# ── CHANGE THESE ──────────────────────────────
year       = "2023"          # ACS year (2009–2023)
variables  = "DP04_0058E"    # variable code(s), comma-separated
state_fips = "18"            # FIPS code for your state (18 = Indiana)
level      = "tract"         # geography: "state", "county", or "tract"
api_key    = ""              # optional — get one free at api.census.gov/data/key_signup.html
# ───────────────────────────────────────────────

## 1.3 — Download the Data

This cell builds the API URL and fetches the data.

In [ ]:
# Build the Census API URL
base_url = f"https://api.census.gov/data/{year}/acs/acs5/profile"
get_cols = f"NAME,GEO_ID,{variables}"

if level == "state":
    url = f"{base_url}?get={get_cols}&for=state:{state_fips}"
elif level == "county":
    url = f"{base_url}?get={get_cols}&for=county:*&in=state:{state_fips}"
else:  # tract
    url = f"{base_url}?get={get_cols}&for=tract:*&in=state:{state_fips}%20county:*"

if api_key:
    url += f"&key={api_key}"

# Fetch and convert to a DataFrame
response = requests.get(url)
data = response.json()
df = pd.DataFrame(data[1:], columns=data[0])  # row 0 is the header

print(f"Downloaded {df.shape[0]} rows and {df.shape[1]} columns")
df.head()

## 1.4 — Save and Reload

We save the data as a CSV file so we don't need to re-download it.

In [ ]:
csv_filename = f"census_{year}_{level}_{state_fips}.csv"

df.to_csv(csv_filename, index=False)
print(f"Saved to: {csv_filename}")

# Reload from file (good habit — verifies the save worked)
df = pd.read_csv(csv_filename)
df.head()

## 1.5 — Clean the GEOID Column

The raw `GEO_ID` looks like `1400000US18001`. For maps we only need the part after `US` (e.g. `18001`).

In [ ]:
df["GEOID"] = df["GEO_ID"].str.split("US").str[1]

print("Sample GEOID values:", df["GEOID"].head().tolist())
df.head()

---
**Part 1 complete.** You have a clean CSV with Census data and a `GEOID` column ready for maps.

# Part 2: Exploring the Data

Now let's look at what we downloaded — column names, data types, and basic statistics.

---

## 2.1 — Load the Data

Update `csv_filename` to match the file name printed in Step 1.4. If using different census tract file manually downloaded then update it with that name.

In [ ]:
# import pandas as pd
#csv_filename = "census_2023_tract_18.csv"   # uncomment and update name if different

df = pd.read_csv(csv_filename)

# Re-create GEOID if needed
if "GEOID" not in df.columns:
    df["GEOID"] = df["GEO_ID"].str.split("US").str[1]

print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2.2 — Inspect the Columns

`.info()` shows column names, data types, and how many values are non-null (not empty).

In [ ]:
df.info()

## 2.3 — Convert Numbers and Check for Missing Values

The Census API sends everything as text. We convert estimate columns to real numbers.

Census also uses `-666666666` as a placeholder for *'no data'* — we replace those with blank (`NaN`).

In [ ]:
CENSUS_NULL = -666666666

# Find all estimate columns (they end with 'E')
estimate_cols = [c for c in df.columns if c.endswith("E") and c not in ("NAME", "GEO_ID", "GEOID")]

# Convert to numbers and replace placeholders
for col in estimate_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].replace(CENSUS_NULL, pd.NA)

# Summary statistics
df[estimate_cols].describe().round(1)

## 2.4 — Quick Look at Your Variable

Update `variable` to whichever column you downloaded.

In [ ]:
variable  = "DP04_0058E"             # update to your variable
# variable2 = "DP...."
# var_label = "Households Without a Vehicle"  # friendly name for charts

col = df[variable]
print(f"Variable : {variable}")
print(f"Min      : {col.min():.0f}")
print(f"Max      : {col.max():.0f}")
print(f"Mean     : {col.mean():.1f}")
print(f"Median   : {col.median():.1f}")
print(f"Missing  : {col.isna().sum()} rows")

---
**Part 2 complete.** You know the shape of your data and the range of your variable.

# Part 3: Mapping Prep — Download Shapefiles and Join

Census data tells us *numbers* per area. Shapefiles tell us the *boundaries* of those areas.  
We download the shapefile and merge the two together.

---

## 3.1 — Import Libraries

In [ ]:
import pandas as pd
import geopandas as gpd
import requests, zipfile, os, shutil
import urllib3
urllib3.disable_warnings()   # suppress SSL warnings when downloading shapefiles

## 3.2 — Load the Cleaned CSV

In [ ]:
# csv_filename = "census_2023_tract_18.csv"   # update if different
variable     = "DP04_0058E"                  # update to your variable or update as list ["DP..., DP..."]
# variable2 = "DP..."
# variable3 = "DP..."

df = pd.read_csv(csv_filename)

if "GEOID" not in df.columns:
    df["GEOID"] = df["GEO_ID"].str.split("US").str[1]

# Convert to numeric and remove placeholders
CENSUS_NULL = -666666666
estimate_cols = [c for c in df.columns if c.endswith("E") and c not in ("NAME","GEO_ID","GEOID")]
for col in estimate_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace(CENSUS_NULL, pd.NA)

df_clean = df.dropna(subset=[variable]).copy()
print(f"{len(df_clean)} rows after removing missing values")

## 3.3 — Download the TIGER Shapefile

Set `fips_code` to match the state you used in Part 1.

In [ ]:
fips_code = 18    # update to your state FIPS
year      = 2023  # match the year from Part 1

file_name = f"tl_{year}_{fips_code:02d}_tract.zip"
url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/{file_name}"

# Download
r = requests.get(url, verify=False)
with open(file_name, "wb") as f:
    f.write(r.content)
print("Downloaded:", file_name)

# Extract
folder = file_name.replace(".zip", "")
with zipfile.ZipFile(file_name, "r") as z:
    z.extractall(folder)
os.remove(file_name)
print("Extracted to:", folder)

## 3.4 — Join Shapefile with Census Data

We match rows using the `GEOID` column — it is the shared key between the two datasets.

In [ ]:
shp_path = os.path.join(folder, f"tl_{year}_{fips_code:02d}_tract.shp")

gdf = gpd.read_file(shp_path)
print(f"Shapefile: {len(gdf)} tracts")

merged_gdf = gdf.merge(df_clean, on="GEOID", how="inner")
merged_gdf[variable] = pd.to_numeric(merged_gdf[variable], errors="coerce")

print(f"After join: {len(merged_gdf)} tracts matched")
merged_gdf.head()

## 3.5 — Top Counties

The first 5 digits of a tract GEOID identify the county. We group by those digits to get county totals.

In [ ]:
merged_gdf["county_fips"] = merged_gdf["GEOID"].str[:5]

county_totals = (
    merged_gdf.groupby("county_fips")[variable]
    .sum()
    .reset_index()
    .rename(columns={variable: "total"})
    .sort_values("total", ascending=False)
)

print("Top 10 counties:")
county_totals.head(10)

---
**Part 3 complete.** You have a GeoDataFrame ready to map.

# Part 4: Visualizing the Data

Time to make maps and charts!

---

## 4.1 — Import Libraries

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

## 4.2 — Choropleth Map

Each census tract is colored by its value. Darker = higher.

In [ ]:
# print(variable) # print if needed
var_label = "Households Without a Vehicle"  # friendly name for charts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

merged_gdf.plot(
    column=variable, # if variable is a list then use variable[0], variable[1]
    cmap="YlOrRd",          # color palette
    linewidth=0.15,
    edgecolor="grey",
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No data"},
    ax=ax
)

ax.set_title(f"{var_label} by Census Tract (2023 ACS)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## 4.3 — Bar Chart: Top 15 Counties

In [ ]:
top15 = county_totals.head(15).sort_values("total")  # sort ascending for horizontal bar

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15["county_fips"], top15["total"], color="steelblue")

ax.set_xlabel(var_label)
ax.set_title(f"Top 15 Counties — {var_label}")
ax.bar_label(bars, fmt="{:,.0f}", padding=4)

plt.tight_layout()
plt.show()

## 4.4 — Histogram

This shows how the values are spread across all tracts. The red line is the median, orange is the mean.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(merged_gdf[variable].dropna(), bins=40, color="steelblue", edgecolor="white")

ax.axvline(merged_gdf[variable].median(), color="red",    linestyle="--",
           label=f"Median: {merged_gdf[variable].median():.0f}")
ax.axvline(merged_gdf[variable].mean(),   color="orange", linestyle="--",
           label=f"Mean:   {merged_gdf[variable].mean():.0f}")

ax.set_xlabel(var_label)
ax.set_ylabel("Number of Census Tracts")
ax.set_title(f"Distribution of {var_label} by Tract")
ax.legend()

plt.tight_layout()
plt.show()

---
**All done!** You have downloaded, cleaned, joined, and visualized Census data.  
Try swapping in a different variable code in Part 1 and running through again to compare.